# OPERA-LM on Kaggle -- a stronger, forkable checkpoint

**This is a PR move, not a research move.** The paper draft and the Colab
demo already make the honest research case (matched-baseline comparisons,
a documented falsification record, an explicit "data-regime dependence"
limitation). This notebook has a narrower goal: **train a better checkpoint
than the current 150M/82M-token demo, on Kaggle's free weekly GPU quota, so
there's something more capable for people to fork, poke at, and improve.**

**Why this matters framed honestly:** OPERA is a few months old, built by
one person in spare time. Transformers have had a decade of institutional
engineering -- fused kernels, distributed training recipes, trillions of
tokens of pretraining runs, thousands of contributors. The gaps you'll see
in this project (slower wall-clock per step despite fewer FLOPs, no
multi-GPU support yet, small data budgets) are exactly what "young
architecture" looks like, not a hidden flaw. If something here looks like
an obvious opportunity to help -- it probably is. See the **"Where a
contributor's afternoon would matter most"** cell near the end for
specifics (kernel work, the CUDA/Triton port of the scan arm, multi-GPU
training, data-mix experiments).

**What this notebook does:**
1. Detects the actual Kaggle accelerator and measures real throughput
   before committing to a schedule (P100 and 2xT4 are *not* just "half
   an A100" or "double a T4" by default -- see the throughput cell).
2. Preps FineWeb-Edu data (optionally mixed with FineMath + Stack-Edu,
   SmolLM2-style) via the existing, unmodified `prepare_fineweb.py`.
3. Trains OPERA with the existing `train_chat.py`, sized to fit one
   Kaggle session's time budget, checkpointing to a **Kaggle Dataset**
   (the Kaggle-native equivalent of Colab's Drive mount -- current
   private-storage quota is 200GB, plenty for repeated checkpoints).
4. On the next session, resumes from that dataset via `--resume`
   (exact batch-stream + RNG state, already built into `train_chat.py`).
5. Publishes to a Hugging Face Space when you're happy with it, via the
   same `push_to_hub.py` the Colab notebook uses.

Repeat steps 3-4 across a few sessions/week (quota resets weekly) until
the checkpoint is where you want it, then publish.


## 0. Detect the accelerator and confirm the environment

Kaggle gives you either one P100 or two T4s, sharing one ~30 GPU-hour/week quota. Don't assume which one you have, or its throughput -- check it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  cuda:{i}', torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

# Confirmed by an actual failed run (2026-08), not a guess: Kaggle's
# prebuilt PyTorch wheel ships NO compiled kernels for compute
# capability 6.0 (P100/Pascal) at all -- every CUDA op fails with
# "no kernel image is available for execution on the device", even a
# plain nn.Embedding lookup outside any compiled region. --compile off
# does NOT fix this. Fail fast instead of burning a smoke-test cycle
# finding out the hard way.
if torch.cuda.is_available() and torch.cuda.get_device_capability(0) < (7, 0):
    raise RuntimeError(
        "This GPU's compute capability is below sm_70 -- Kaggle's PyTorch "
        "build has no kernels for it (confirmed: P100/sm_60 fails on a "
        "bare nn.Embedding). Switch accelerator to 'GPU T4 x2' in this "
        "notebook's Settings and restart the session -- do not continue "
        "past this cell on a P100."
    )


**Reading the output:** compute capability `(8, 0)`+ is Ampere-class (bf16, well-supported by `torch.compile`). `(7, 5)` is a T4 (Turing -- fp16+GradScaler path, well-supported). `(6, 0)` is a **P100** (Pascal, 2016) -- **confirmed broken**, not just old: Kaggle's PyTorch build has no compiled kernels for it at all, so every CUDA op fails outright (verified by an actual run -- see the cell above's hard check). This isn't a `torch.compile`/Triton limitation you can route around with `--compile off`; there's no working code path on a P100 with this environment. If you see `(6, 0)`, the cell above already stops you -- switch accelerator to `GPU T4 x2` and restart the session.

If you got 2x T4 (`device count: 2`): `opera_lm.train.train()` now has DDP support (§5's optional multi-GPU cell) so both GPUs can actually be used, not just one with the second idle. Honest status: verified end-to-end on CPU/gloo, not yet on real multi-GPU hardware -- your run here is the first real test of it, see that cell's note.


## 1. Clone + install

Same as the Colab notebook -- Kaggle notebooks clone from GitHub the same way.

In [ ]:
REPO_URL = "https://github.com/Merna-Khalid/OPERA-LM"
!git clone $REPO_URL repo
%cd repo
!pip install -q datasets tokenizers huggingface_hub


## 2. Persistent storage: a Kaggle Dataset for checkpoints

Kaggle has no Drive-style mount. The equivalent pattern: push your
checkpoint directory as a new **version** of a Kaggle Dataset at the end
of each session, then attach that dataset as a read-only input at the
start of the next one.

**One-time setup (do this once, outside this notebook):**
1. kaggle.com -> Settings -> API -> "Create New Token" -> downloads
   `kaggle.json` (contains your username + key).
2. In this notebook's editor: **Add-ons -> Secrets** -> add two secrets,
   `KAGGLE_USERNAME` and `KAGGLE_KEY`, with the values from that file.
   (Kaggle's own API needs its own credentials passed explicitly like
   this, even though you're already logged into Kaggle to run the
   notebook -- the `kaggle` CLI package doesn't inherit that session.)

Run the cell below every session -- cheap, and makes the later
`!kaggle datasets version` calls work.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = secrets.get_secret("KAGGLE_KEY")
!pip install -q kaggle

CKPT_DATASET_SLUG = f"{os.environ['KAGGLE_USERNAME']}/opera-lm-checkpoints"
WORK_DIR = "/kaggle/working/opera_ckpt"
os.makedirs(WORK_DIR, exist_ok=True)
print("checkpoint dataset:", CKPT_DATASET_SLUG)
print("work dir:", WORK_DIR)


### Resume check: pull the latest checkpoint in, if one exists

First session ever: this finds nothing, and that's fine -- you'll create
the dataset for the first time in the "push back" cell later. Every
session after that: attach `CKPT_DATASET_SLUG` as a notebook input
(**Add Input -> search your username -> opera-lm-checkpoints**) before
running this cell, so `/kaggle/input/opera-lm-checkpoints/` exists.

In [ ]:
import glob, shutil

INPUT_CKPT_DIR = "/kaggle/input/opera-lm-checkpoints"
RESUME = os.path.isdir(INPUT_CKPT_DIR) and len(os.listdir(INPUT_CKPT_DIR)) > 0

if RESUME:
    for f in glob.glob(f"{INPUT_CKPT_DIR}/*"):
        dst = os.path.join(WORK_DIR, os.path.basename(f))
        if os.path.isdir(f):
            shutil.copytree(f, dst, dirs_exist_ok=True)
        else:
            shutil.copy(f, dst)
    print(f"RESUMING: copied {os.listdir(WORK_DIR)} from {INPUT_CKPT_DIR}")
else:
    print("No existing checkpoint dataset attached -- starting fresh this session.")


## 3. Throughput smoke test (measure, don't assume)

30 steps, real training config, `torch.compile` on. This tells you your
*actual* tokens/second on whatever accelerator you got -- the number the
rest of this notebook's step budget is computed from, not a guess.

In [ ]:
!python opera-chat/prepare_fineweb.py --smoke \
  --tokenizer opera-chat/tokenizer.json --out /kaggle/working/data_smoke.pkl

!python opera-chat/train_chat.py \
  --data /kaggle/working/data_smoke.pkl \
  --steps 30 --batch 16 --max-len 256 --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda --optimizer muon --muon-lr 0.02 \
  --compile default --out-dir /kaggle/working/smoke --save-every 0


**Read the `s/step` line at `T_cur 256`** (the steady-state, full-length
number -- early curriculum steps at T_cur 64 are faster and not
representative). Compare against the reference points:

| accelerator | measured elsewhere | tokens/s (batch16 x 256) |
|---|---|---|
| A100-SXM4-40GB (Colab, this project's own number) | 0.65 s/step | ~6,300 |
| P100 / T4, *if* `torch.compile` works normally | untested here -- this is what you're measuring | ? |

If the smoke test errored inside `torch._dynamo`/Triton (not a normal
Python exception), rerun with `--compile off` and use *that* number
instead -- it means Triton doesn't support this GPU generation well, and
eager mode is your real baseline, not a bug to chase.

**Set `MEASURED_S_PER_STEP` below** to the number you actually got, so
the training step budget in the next section is sized off reality:

In [ ]:
MEASURED_S_PER_STEP = 0.65   # <-- replace with your smoke-test number


## 3b. Curriculum / length OOM smoke test (before the real data prep)

The throughput smoke test above only ran 30 steps at the curriculum's
starting length (`T_cur=64`, default `--curriculum-every 250`), so it
never actually exercises `T_cur=128` or `T_cur=256` -- and those are
exactly the lengths that OOM'd on a real run (`compose_pair_batch` at
`T_cur=128`, then `msup_loss` one step later, both now fixed via
`--grad-checkpoint level` and the `aux_frac` wiring into `msup_loss`).
This cell reuses the same 5M-token `data_smoke.pkl` from above (no new
download) but forces a fast curriculum (`--curriculum-every 20`) so it
ramps `T_cur: 64 -> 128 -> 256` inside ~60 steps -- a couple minutes,
not hours -- and matches whichever path (single-GPU or DDP) section 5
will actually run. Confirm this finishes clean (or at least clears
`T_cur=256` a few steps) before spending time on the full 2B-token
streaming data prep in section 4.


In [ ]:
N_GPUS = torch.cuda.device_count()
CURRICULUM_TEST_STEPS = 60   # enough to reach T_cur=256 and run a few steps there
CURRICULUM_TEST_EVERY = 20   # T_cur: 64 (step 0) -> 128 (step 20) -> 256 (step 40)

if N_GPUS >= 2:
    print(f"Testing the DDP path across {N_GPUS} GPUs (matches section 5's multi-GPU cell)")
    !torchrun --nproc_per_node=$N_GPUS opera-chat/train_chat.py \
      --ddp \
      --data /kaggle/working/data_smoke.pkl \
      --steps $CURRICULUM_TEST_STEPS \
      --batch 16 --max-len 256 --eval-max-len 2048 \
      --d 1664 --nb 416 --num-layers 8 \
      --device cuda --optimizer muon --muon-lr 0.02 \
      --compile off --grad-checkpoint level \
      --curriculum-t0 64 --curriculum-every $CURRICULUM_TEST_EVERY \
      --out-dir /kaggle/working/len_test --save-every 0
else:
    print("Testing the single-GPU path (matches section 5's single-GPU cell)")
    !python opera-chat/train_chat.py \
      --data /kaggle/working/data_smoke.pkl \
      --steps $CURRICULUM_TEST_STEPS \
      --batch 16 --max-len 256 --eval-max-len 2048 \
      --d 1664 --nb 416 --num-layers 8 \
      --device cuda --optimizer muon --muon-lr 0.02 \
      --compile default --grad-checkpoint level \
      --curriculum-t0 64 --curriculum-every $CURRICULUM_TEST_EVERY \
      --out-dir /kaggle/working/len_test --save-every 0


## 4. Data prep

Plain FineWeb-Edu by default (zero new risk, reuses the exact pipeline
validated on Colab). The SmolLM2-style mix (FineWeb-Edu 85% / FineMath
10% / Stack-Edu 5%) is one flag away if you want it -- both paths use
the identical tokenize/chunk code downstream, since all three datasets
expose a plain `"text"` field.

In [ ]:
DATA_FW = "/kaggle/working/data_fineweb.pkl"

# Plain FineWeb-Edu (default, recommended to start):
!python opera-chat/prepare_fineweb.py \
  --tokenizer opera-chat/tokenizer.json \
  --out $DATA_FW \
  --max-tokens 2000000000 \
  --max-len 256 --eval-max-len 2048

# SmolLM2-style mix instead -- uncomment to use:
# !python opera-chat/prepare_fineweb.py \
#   --tokenizer opera-chat/tokenizer.json \
#   --out $DATA_FW \
#   --mix fineweb,finemath,stackedu --mix-weights 0.85,0.10,0.05 \
#   --max-tokens 2000000000 --max-len 256 --eval-max-len 2048


## 5. Train, sized to this session's time budget

Kaggle sessions cap at ~12 hours; leave headroom for the smoke test,
data prep, and the checkpoint push at the end. `TARGET_HOURS=10` below
is a conservative default -- tighten it once you've run a full session
and know your actual per-session overhead.

In [ ]:
TARGET_HOURS = 10
STEPS = int(TARGET_HOURS * 3600 / MEASURED_S_PER_STEP)
print(f"Training for {STEPS} steps (~{TARGET_HOURS}h at {MEASURED_S_PER_STEP}s/step)")

RESUME_FLAG = "--resume" if RESUME else ""

!python opera-chat/train_chat.py \
  --data $DATA_FW \
  --steps $STEPS \
  --batch 16 --max-len 256 --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda --optimizer muon --muon-lr 0.02 \
  --compile default --save-every 1000 \
  --grad-checkpoint level \
  --out-dir $WORK_DIR $RESUME_FLAG


### Optional: multi-GPU (2x T4), if you got that accelerator

`opera_lm.train.train()` now has real DDP support (added 2026-08-16,
after measuring single-T4 throughput and deciding the 2x-idle-GPU gap
was worth closing). **Honest status: verified end-to-end on CPU/gloo via
`opera_lm.selftest` (both the `aux_frac` and `msup` code paths, which
each call the LM head a second time out-of-band and needed
`find_unused_parameters=True` to work under DDP's gradient bucketing --
caught by that test, not assumed). It has NOT been run on real multi-GPU
hardware.** This machine has none to test against; Kaggle's 2xT4 is the
first real test. If it breaks, the single-GPU cell above is the
reliable fallback -- report back what happened either way.

Notes: `--batch` is *per-GPU* (effective global batch = batch x 2 here).
`torch.compile` is force-disabled under `--ddp` (stacking two
individually-already-tricky risk surfaces -- compile bugs have hit this
codebase twice this session already -- isn't worth it for a first pass).
`STEPS` below reuses the single-GPU `MEASURED_S_PER_STEP` as a
per-step-time estimate, which is conservative (DDP processes 2x the
tokens per step for similar-or-slightly-higher per-step wall time, so
this likely *underestimates* how many steps fit in the budget -- safer
to finish early than to overrun the session cap).

In [ ]:
N_GPUS = torch.cuda.device_count()
if N_GPUS < 2:
    print(f"Only {N_GPUS} GPU(s) visible -- skipping, use the single-GPU cell above.")
else:
    TARGET_HOURS_DDP = 10
    STEPS_DDP = int(TARGET_HOURS_DDP * 3600 / MEASURED_S_PER_STEP)
    print(f"Training for {STEPS_DDP} steps across {N_GPUS} GPUs "
          f"(~{TARGET_HOURS_DDP}h at the single-GPU {MEASURED_S_PER_STEP}s/step "
          f"estimate; effective batch {16 * N_GPUS})")

    RESUME_FLAG = "--resume" if RESUME else ""

    !torchrun --nproc_per_node=$N_GPUS opera-chat/train_chat.py \
      --ddp \
      --data $DATA_FW \
      --steps $STEPS_DDP \
      --batch 16 --max-len 256 --eval-max-len 2048 \
      --d 1664 --nb 416 --num-layers 8 \
      --device cuda --optimizer muon --muon-lr 0.02 \
      --compile off --save-every 1000 \
      --grad-checkpoint level \
      --out-dir $WORK_DIR $RESUME_FLAG


## 6. Push the checkpoint back to the Kaggle Dataset

**First time ever** (dataset doesn't exist yet): needs a
`dataset-metadata.json` in `WORK_DIR` first, created once by
`kaggle datasets init`, then `kaggle datasets create`.
**Every session after that:** `kaggle datasets version`.

In [ ]:
import json as _json

meta_path = f"{WORK_DIR}/dataset-metadata.json"
if not os.path.exists(meta_path):
    !kaggle datasets init -p $WORK_DIR
    with open(meta_path) as f:
        meta = _json.load(f)
    meta["title"] = "opera-lm-checkpoints"
    meta["id"] = CKPT_DATASET_SLUG
    with open(meta_path, "w") as f:
        _json.dump(meta, f, indent=2)
    !kaggle datasets create -p $WORK_DIR --dir-mode zip
else:
    !kaggle datasets version -p $WORK_DIR -m "step $STEPS" --dir-mode zip


## 7. Repeat next session

Weekly quota resets, session cap is ~12h -- realistically 2-3 sessions/week.
Each session: **Add Input** the `opera-lm-checkpoints` dataset (now
containing last session's checkpoint), run this notebook top to bottom
again. Cells 0-2 are cheap and idempotent; cell 2's resume check will
find the attached dataset and pick up where you left off via `--resume`
(exact batch-stream + RNG state -- already built into `train_chat.py`,
nothing extra needed here).

## 8. Publish when ready

Same `push_to_hub.py` the Colab notebook uses -- see `opera-chat/README.md`
for the full deploy-artifacts contract. Both the model repo and a CPU
Basic Space are free on Hugging Face regardless of model size.

In [ ]:
import glob, os
ckpts = [f for f in glob.glob(f'{WORK_DIR}/*.pt') if not f.endswith('_train_ckpt.pt')]
FINAL_CKPT = max(ckpts, key=os.path.getmtime)
print('checkpoint:', FINAL_CKPT)

!python opera-chat/push_to_hub.py \
  --ckpt $FINAL_CKPT \
  --config $WORK_DIR/model_config.json \
  --model-repo USER/opera-lm-chat-strong \
  --space-repo USER/opera-lm-chat-strong-space


## Where a contributor's afternoon would matter most

If you're reading this because the demo pointed you here: these are the
concrete, scoped gaps a decade of transformer engineering has already
solved for that architecture and OPERA hasn't gotten to yet. Not a vague
"contributions welcome" -- pick one:

- **Harden the new DDP path**: `opera_lm.train.train(ddp=True)` (added
  2026-08-16) is verified on CPU/gloo via `opera_lm.selftest` but not yet
  on real multi-GPU hardware -- §5's optional cell is the first real
  test. If it breaks on your 2xT4 run, or if you push it further (e.g.
  `torch.compile` under DDP, currently force-disabled -- see the
  docstring note at the top of `train()`), that's directly useful.
- **Fused CUDA/Triton composition kernels**: the compose node is
  currently many small bandwidth-bound PyTorch ops; a Metal prototype
  exists (`opera_lm.metal_kernel`) but there's no CUDA equivalent. This
  is the single biggest lever on the ~8x per-step slowdown vs a matched
  transformer at this model size (see the paper's §7, "Speed").
- **The associative-scan arm's CUDA path**: `fold_mode='scan'` (an
  associative-scan sibling to tree+fold, see `docs/OPERA_Scan_Arm_Design.md`)
  already validates at quality parity with `left` at 0.57x the op
  dispatches on CPU/MPS -- it's never been run on CUDA.
- **Data-mix experiments**: the `--mix` flag on `prepare_fineweb.py`
  (added alongside this notebook) is wired up but unvalidated -- does
  FineMath/Stack-Edu actually help a from-scratch 150M model the way it
  does SmolLM2's much bigger training runs? Nobody's run that ablation
  yet.
- **Multi-seed replication**: every result in `docs/OPERA_Paper_Draft_v1.3.md`
  is single-seed. Seeds 1-2 on the main arms is listed as a `[TODO]`
  throughout the paper, not because it's unimportant, but because it's
  genuinely expensive on one person's compute budget.

Repo: https://github.com/Merna-Khalid/OPERA-LM -- start with
`docs/OPERA_Mechanisms_Guide.md` and `opera_lm/selftest.py` (the
self-test suite everything here is checked against).
